In [ ]:
# =========================================
# 0) Imports / Environment
# =========================================
import os
import re
import ast
import math
from pathlib import Path

import pandas as pd
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    LogitsProcessor,
    LogitsProcessorList,
)

# ---- (서버 기준) 경로는 필요에 따라 수정 ----
XLSX_PATH = "/home/rsm/miniconda3/llm_prac/dummy.xlsx"   # 서버 실제 경로
OUT_DIR   = "/home/rsm/DATE_OUT"

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

# HF 캐시 경로(당신 요약대로 /home/rsm 기준 통일)
os.environ.setdefault("HF_HOME", "/home/rsm/hf_home")
os.environ.setdefault("TRANSFORMERS_CACHE", "/home/rsm/hf_cache")
os.environ.setdefault("HF_DATASETS_CACHE", "/home/rsm/hf_datasets_cache")

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch.cuda.is_available():", torch.cuda.is_available())
print("device:", device)
print("HF_HOME:", os.environ.get("HF_HOME"))
print("TRANSFORMERS_CACHE:", os.environ.get("TRANSFORMERS_CACHE"))

torch.cuda.is_available(): True
device: cuda
HF_HOME: /home/rsm/hf_home
TRANSFORMERS_CACHE: /home/rsm/hf_cache


In [7]:
# =========================================
# 1) Load model/tokenizer (Qwen2.5 - HF chat template)
# =========================================
torch.set_grad_enabled(False)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# (권장) left padding + pad_token 지정 (배치/생성 안정)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="auto" if device == "cuda" else None,
)

# Qwen2.5는 eos_token_id만 써도 충분 (Llama의 eot_id terminator 제거)
EOS_ID = tokenizer.eos_token_id

print("pad_token_id:", tokenizer.pad_token_id)
print("eos_token_id:", tokenizer.eos_token_id)
print("device:", model.device)

model.eval()

pad_token_id: 151643
eos_token_id: 151645
device: cuda:0


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((3584,), eps=1e-06)
    (ro

In [4]:
df = pd.read_excel(XLSX_PATH)
assert "key" in df.columns and "text_body" in df.columns

def to_text(x):
    if pd.isna(x):
        return ""
    if isinstance(x, list):
        return " ".join(map(str, x))
    s = str(x).strip()
    if s.startswith("[") and s.endswith("]"):
        try:
            v = ast.literal_eval(s)
            if isinstance(v, list):
                return " ".join(map(str, v))
        except:
            pass
    return s

df["text_norm"] = df["text_body"].apply(to_text)
print(df.shape)

(5, 4)


In [6]:
import re
import pandas as pd
from pathlib import Path

# 저장 경로 (원하는 곳으로 바꿔도 됨)
OUT_DIR = Path("./outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

date_re = re.compile(r"(\d{4}년|\d{1,2}월|\d{1,2}일|\d+개월|올해|내년|작년)")

debug_rows = []  # 온점 분리 결과 전체 저장용
rows = []        # 날짜 포함 문장만 저장용 (기존 기능)

for _, r in df.iterrows():
    key = r["key"]

    # 네 요구대로 text_body만 쓰려면 아래 한 줄을 text_body로 바꾸면 됨:
    # text = r.get("text_body", "")
    text = r.get("text_norm", "")

    if not isinstance(text, str) or not text.strip():
        continue

    # 온점(.) 기준 문장 분리
    sents = re.split(r"\.[\s\n]*", text)

    for idx, s in enumerate(sents, start=1):
        s = (s or "").strip()
        if not s:
            continue

        has_date = bool(date_re.search(s))

        # 1) 디버그 저장: 온점 분리된 "전체 문장"을 저장
        debug_rows.append({
            "key": key,
            "sent_idx": idx,          # 문서 내 몇 번째 문장인지
            "sentence": s,            # 분리된 문장
            "has_date": has_date      # 날짜 패턴 포함 여부
        })

        # 2) 기존 로직: 날짜 포함 문장만 따로 모으기
        if has_date:
            rows.append({"key": key, "date_sentence": s})

# 디버그용 CSV (온점 분리 확인)
split_debug_df = pd.DataFrame(debug_rows)
split_debug_path = OUT_DIR / "sentences_split_debug.csv"
split_debug_df.to_csv(split_debug_path, index=False, encoding="utf-8-sig")
print("saved split_debug:", split_debug_df.shape, "->", split_debug_path)

# 날짜 문장 CSV (기존)
date_df = pd.DataFrame(rows).drop_duplicates()
date_path = OUT_DIR / "date_sentences.csv"
date_df.to_csv(date_path, index=False, encoding="utf-8-sig")
print("date_df:", date_df.shape, "->", date_path)

# 빠른 확인(상위 10개)
split_debug_df.head(10), date_df.head(10)

saved split_debug: (526, 4) -> outputs/sentences_split_debug.csv
date_df: (33, 2) -> outputs/date_sentences.csv


(      key  sent_idx                                           sentence  \
 0  772109         1  심 리 평 가  보 고 서 PSYCHOLOGICAL ASSESSMENT REPORT...   
 1  772109         2  의뢰사유 및 주호소      상기 환자는 교우 문제와 가족 내 갈등으로 우울과 불안...   
 2  772109         3  2018년 2월 27일에 군입대 신체검사를 앞두고 면밀한 평가를 위해서 본원 정신건...   
 3  772109         4  2018년 5월에 병역판정검사를 하여 1급 현역 판정을 받았으며 2020년 8월로 ...   
 4  772109         5               입대를 앞두고 현재의 정신상태와 관련해 정밀평가를 받고자 재내원함   
 5  772109         6                                 이에 종합심리평가 의뢰하여 실시함   
 6  772109         7  환자와의 면담에 의하면, 작년 5월에 실시한 병역판정검사에서 1급 현역 판정을 받았...   
 7  772109         8                                    군입대를 앞두고 걱정이 많음   
 8  772109         9  체력이 약해서 강도 높은 훈련을 잘 버틸 수 있을지 자신이 없고, 무엇보다도 군대라...   
 9  772109        10  올해 초부터 교내 기독교 동아리에서 지원해주는 기숙사에서 형들과 공동체 생활을 하고...   
 
    has_date  
 0     False  
 1      True  
 2      True  
 3      True  
 4     False  
 5     False  
 6      True  
 7     False  
 8     False  
 9      True  ,
       

In [8]:

CLS_SYSTEM = """
너는 주어진 문장에서 '스트레스에 영향을 끼칠 사건이 포함된 문장'을 분류한다.

라벨 정의:
- T:  '스트레스 사건'이 문장에 명시됨. 이는 아래 범주에 포함되는 문장을 뜻함. 
    - 갑질/권력남용: 상사/권한을 이용한 부당대우, 강요, 업무 괴롭힘, 통제
    - 폭행/신체위협: 때림, 밀침, 상해, 물리적 위협
    - 비난/모욕/언어폭력: 욕설, 조롱, 모욕, 공개적 비난, 비꼼
    - 잔소리/통제/간섭: 반복 잔소리, 과도한 간섭, 생활 통제
    - 의심/질투/감시: 의심, 질투, 감시, 확인, 폰검사 등으로 갈등 유발
    - 괴롭힘/따돌림/협박: 따돌림, 괴롭힘, 협박, 위협, 스토킹
    - 가족/부부갈등: 가족/부부 사이의 다툼, 갈등, 폭언/통제 등이 사건으로 서술됨
    - 기타(대인스트레스): 위 범주에 딱 안 맞지만 대인 스트레스 사건으로 볼 수 있음
- F: 사건이 명시되지 않음(검사/의뢰/내원/예약/입대/학년/일정/배경정보 등 메타) 또는
     단순 감정(힘듦/우울/불안/스트레스)만 있고 '사건'이 구체적으로 나오지 않음.
- U: 문장만으로 사건 여부를 판단하기 어려움(주어/사건이 불명확하거나 문장 자체가 너무 단편적).

규칙:
- 애매하면 U가 아니라 F로 답해라. (과잉 탐지 방지)
- 출력은 반드시 한 글자만: T 또는 F 또는 U.
""".strip()

CLS_FEWSHOT = [
    ("문장: 2024년 3월 2일 심리평가 의뢰됨.\n출력:", "F"),
    ("문장: 2025년 8월 입대 예정임.\n출력:", "F"),
    ("문장: 2019년 2학년 때 전학함.\n출력:", "F"),
    ("문장: 작년부터 스트레스를 많이 받았다고 함.\n출력:", "F"),
    ("문장: 올해 1월 외래 내원하여 상담을 받음.\n출력:", "F"),

    ("문장: 작년 11월 상사로부터 지속적인 갑질을 당했다고 함.\n출력:", "T"),
    ("문장: 2023년 5월 폭행을 당한 이후 불안이 심해졌다고 함.\n출력:", "T"),
    ("문장: 지난달 가족에게 지속적인 비난을 들으며 갈등이 심해졌다고 함.\n출력:", "T"),
    ("문장: 올해 초 배우자의 의심과 통제로 심한 다툼이 반복되었다고 함.\n출력:", "T"),
    ("문장: 최근 몇 달간 잦은 잔소리와 간섭으로 가정 내 갈등이 커졌다고 함.\n출력:", "T"),
]

def classify_one(sentence: str):
    msgs = [{"role": "system", "content": CLS_SYSTEM}]
    for u, a in CLS_FEWSHOT:
        msgs.append({"role": "user", "content": u})
        msgs.append({"role": "assistant", "content": a})
    msgs.append({"role": "user", "content": f"문장: {sentence}\n출력:"})

    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    out = model.generate(
        **inputs,
        max_new_tokens=1,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,   # ✅ 여기만 변경
    )
    gen = tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    first = (gen.strip()[:1] if gen else "")
    label = first if first in ["T", "F", "U"] else "F"  # 이상하면 F(보수)
    return label, gen

cls_rows = []
for i, r in date_df.iterrows():
    label, raw = classify_one(r["date_sentence"])
    cls_rows.append({"key": r["key"], "date_sentence": r["date_sentence"], "label": label, "_raw": raw})
    if (i+1) % 50 == 0:
        print("processed", i+1)

cls_df = pd.DataFrame(cls_rows)
cls_df["label"].value_counts()


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


label
F    16
T    14
U     3
Name: count, dtype: int64

In [9]:
cls_rows = []
for i, r in date_df.iterrows():
    label, raw = classify_one(r["date_sentence"])
    cls_rows.append({
        "key": r["key"],
        "date_sentence": r["date_sentence"],
        "label": label,
        "_raw": raw,
    })
    if (i+1) % 50 == 0:
        print("processed", i+1)

cls_df = pd.DataFrame(cls_rows)
print(cls_df["label"].value_counts())
cls_df.head(10)

label
F    16
T    14
U     3
Name: count, dtype: int64


,key,date_sentence,label,_raw
0,772109,의뢰사유 및 주호소 상기 환자는 교우 문제와 가족 내 갈등으로 우울과 불안...,F,F
1,772109,2018년 2월 27일에 군입대 신체검사를 앞두고 면밀한 평가를 위해서 본원 정신건...,F,F
2,772109,2018년 5월에 병역판정검사를 하여 1급 현역 판정을 받았으며 2020년 8월로 ...,F,F
3,772109,"환자와의 면담에 의하면, 작년 5월에 실시한 병역판정검사에서 1급 현역 판정을 받았...",F,F
4,772109,올해 초부터 교내 기독교 동아리에서 지원해주는 기숙사에서 형들과 공동체 생활을 하고...,U,U
5,772109,어머니도 잔소리가 많은 편으로 올해 들어 학업 문제와 관련해 잔소리를 많이 했고 본...,T,T
6,6701677,의뢰사유 및 현병력 2019년 12월 22일 전선에 줄을 매고 자살을 시도하여 본원...,T,T
7,6701677,어느 날(1993년) 배우자와 남자가 방에 함께 있는 것을 보고 난 이후부터 외도를...,T,T
8,6701677,2005년까지 아내가 바람을 피웠다고 생각하며 이에 대해 계속 생각나고 화가 나서 ...,T,T
9,6701677,간이정신건강상태검사에서의 2018년~2019년 결과에서 인지기능이 감퇴 되었다는 뚜...,F,F


In [ ]:
"""
#빡센 버전
EVI_SYSTEM = """
문장에서 '스트레스 사건(T)'의 근거를 뽑아 정리한다.

출력 형식(반드시 1줄만):
사건유형 | 근거표현

절대 규칙:
- 출력은 딱 1줄만. 설명/인사/추가 문장/마무리 멘트 금지.
- 근거표현은 입력 문장에 실제로 존재하는 단어/짧은 구절을 그대로 복사한다. (1~15자 권장, 최대 2개)
- 추론/요약/새 문장 생성 금지. 입력에 없는 말을 만들지 마라.
- 근거를 못 뽑으면 아래 1줄만 출력: 근거없음(판단불가) | NONE
- 단답식으로 대답한다.

사건유형은 아래 중 하나만 선택:
- 갑질/권력남용
- 폭행/신체위협
- 비난/모욕/언어폭력
- 잔소리/통제/간섭
- 의심/질투/감시
- 괴롭힘/따돌림/협박
- 가족/부부갈등
- 기타(대인스트레스)
- 근거없음(판단불가)
""".strip()

EVI_FEWSHOT = [
  ("문장: 2019년 12월 22일에 목을 매는 자살을 시도하여 응급실에 내원함.\n출력:", "폭행/신체위협 | 목을 매는"),
  ("문장: 남편과 다툼 중 칼에 목이 찔리는 상해를 입음.\n출력:", "폭행/신체위협 | 칼에 목이 찔리는"),
  ("문장: 배우자의 외도를 의심함.\n출력:", "의심/질투/감시 | 외도를 의심"),
  ("문장: 심리평가 의뢰됨.\n출력:", "근거없음(판단불가) | NONE"),
]

def extract_evidence(sentence: str, retry: int = 1):
    def run(extra_note=""):
        msgs = [{"role": "system", "content": EVI_SYSTEM + (("\n" + extra_note) if extra_note else "")}]
        for u, a in EVI_FEWSHOT:
            msgs.append({"role": "user", "content": u})
            msgs.append({"role": "assistant", "content": a})
        msgs.append({"role": "user", "content": f"문장: {sentence}\n출력:"})

        prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        out = model.generate(
            **inputs,
            max_new_tokens=24,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=terminators,  # ✅ 여기만 변경
        )
        text = tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()

        line = None
        for ln in text.splitlines():
            ln = ln.strip()
            if "|" in ln:
                line = ln
                break

        if line is None:
            return "근거없음(판단불가) | NONE"

        return line

    line = run()
    if "|" not in line:
        return "UNK", "NONE", line

    cat, quote = [x.strip() for x in line.split("|", 1)]
    if cat == "근거없음(판단불가)" or quote == "NONE" or quote not in sentence:
        return "UNK", "NONE", line

    return cat, quote, line

final_df = cls_df.copy()
final_df["category"] = ""
final_df["evidence_quote"] = ""
final_df["_evi_raw"] = ""

t_idx = final_df.index[final_df["label"] == "T"].tolist()
for k, idx in enumerate(t_idx, start=1):
    s = final_df.at[idx, "date_sentence"]
    cat, quote, raw = extract_evidence(s, retry=1)
    final_df.at[idx, "category"] = cat
    final_df.at[idx, "evidence_quote"] = quote
    final_df.at[idx, "_evi_raw"] = raw

    # 근거 없으면 T 유지하지 않음
    if cat == "UNK":
        final_df.at[idx, "label"] = "U"

    if k % 50 == 0:
        print("evidence processed", k)

final_df["label"].value_counts()
"""

label
U    15
F    12
T     6
Name: count, dtype: int64

In [10]:
#느슨한버전
EVI_SYSTEM = """
문장에서 '스트레스 사건(T)'의 근거를 뽑아 정리한다.

출력 형식(반드시 1줄만):
사건유형 | 근거표현

규칙(실패 줄이기 버전):
- 출력은 반드시 1줄만. 다른 말/설명/접두어(사건유형:, 근거:) 절대 쓰지 마라.
- 사건유형은 아래 8개 중 하나만 고른다.
  단, 딱 맞는 범주를 고르기 어렵거나 애매하면 '기타'로 출력해라.
  단, 기타를 선택할 시에는 다른 사건유형에 속하지 않아야하며 유형이 모호해야한다.
- 근거표현은 입력 문장에 실제로 존재하는 **연속된 문자열을 그대로 복사**한다.
  (추론/요약/바꿔쓰기/띄어쓰기 수정 금지. 입력에 없는 말 금지)
- 근거표현은 1개만 고른다. (| 추가 금지)
- 근거를 못 뽑으면 아래 1줄만 출력:
  근거없음(판단불가) | NONE
- 사건유형 분류시 반드시 사건유형과 일치할 필요가 없으며 의미가 유사하다면 동일 사건유형으로 분류한다.

사건유형 목록:
- 갑질/권력남용
- 폭행/신체위협
- 비난/모욕/언어폭력
- 잔소리/통제/간섭
- 의심/질투/감시
- 괴롭힘/따돌림/협박
- 가족/부부갈등
- 기타
""".strip()

EVI_FEWSHOT = [
  ("문장: 2019년 12월 22일에 목을 매는 자살을 시도하여 응급실에 내원함.\n출력:", "기타 | 목을 매는 자살을 시도"),
  ("문장: 남편과 다툼 중 칼에 목이 찔리는 상해를 입음.\n출력:", "폭행/신체위협 | 칼에 목이 찔리는"),
  ("문장: 배우자의 외도를 의심함.\n출력:", "의심/질투/감시 | 외도를 의심"),
  ("문장: 최근 잦은 잔소리와 간섭으로 갈등이 커졌다고 함.\n출력:", "잔소리/통제/간섭 | 잦은 잔소리"),
  ("문장: 심리평가 의뢰됨.\n출력:", "근거없음(판단불가) | NONE"),
]

def extract_evidence(sentence: str, retry: int = 2):
    def run(extra_note=""):
        msgs = [{"role": "system", "content": EVI_SYSTEM + (("\n" + extra_note) if extra_note else "")}]
        for u, a in EVI_FEWSHOT:
            msgs.append({"role": "user", "content": u})
            msgs.append({"role": "assistant", "content": a})
        msgs.append({"role": "user", "content": f"문장: {sentence}\n출력:"})

        prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        out = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,  # HF 권장 terminators
        )
        text = tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()

        # 첫 줄만 사용(불필요한 줄바꿈 방지)
        line = text.splitlines()[0].strip() if text else ""
        if "|" not in line:
            return "근거없음(판단불가) | NONE"
        return line

    line = run()
    if "|" not in line:
        return "UNK", "NONE", line

    # 파이프가 여러 개면 앞의 2개만 사용(조금 더 관대)
    parts = [x.strip() for x in line.split("|")]
    cat = parts[0] if len(parts) > 0 else "UNK"
    quote = parts[1] if len(parts) > 1 else "NONE"

    # 근거없음 처리
    if "근거없음" in cat or quote == "NONE":
        return "UNK", "NONE", line

    # substring 검증이 핵심(여기서 강등됨) → 실패하면 재시도
    if quote not in sentence:
        for _ in range(retry):
            line2 = run(extra_note="근거표현은 반드시 입력 문장에 있는 문자열을 그대로 복사해야 한다. 출력은 1줄 '사건유형 | 근거표현'.")
            parts2 = [x.strip() for x in line2.split("|")]
            if len(parts2) >= 2:
                cat2, quote2 = parts2[0], parts2[1]
                if ("근거없음" not in cat2) and quote2 != "NONE" and (quote2 in sentence):
                    return cat2, quote2, line2
        return "UNK", "NONE", line

    return cat, quote, line

final_df = cls_df.copy()
final_df["category"] = ""
final_df["evidence_quote"] = ""
final_df["_evi_raw"] = ""

t_idx = final_df.index[final_df["label"] == "T"].tolist()
for k, idx in enumerate(t_idx, start=1):
    s = final_df.at[idx, "date_sentence"]
    cat, quote, raw = extract_evidence(s, retry=2)

    final_df.at[idx, "category"] = cat
    final_df.at[idx, "evidence_quote"] = quote
    final_df.at[idx, "_evi_raw"] = raw

    # 근거 없으면 T 유지하지 않음(기존 로직 유지)
    if cat == "UNK":
        final_df.at[idx, "label"] = "U"

    if k % 50 == 0:
        print("evidence processed", k)

final_df["label"].value_counts()

label
F    16
T    13
U     4
Name: count, dtype: int64

In [11]:
final_df = cls_df.copy()
final_df["category"] = ""
final_df["evidence_quote"] = ""
final_df["_evi_raw"] = ""

t_idx = final_df.index[final_df["label"] == "T"].tolist()

for k, idx in enumerate(t_idx, start=1):
    s = final_df.at[idx, "date_sentence"]
    cat, quote, raw = extract_evidence(s, retry=1)
    final_df.at[idx, "category"] = cat
    final_df.at[idx, "evidence_quote"] = quote
    final_df.at[idx, "_evi_raw"] = raw

    # 근거를 못 뽑으면(UNK|NONE) -> T가 아니라 U로 강등
    if cat == "UNK":
        final_df.at[idx, "label"] = "U"

    if k % 50 == 0:
        print("evidence processed", k)

print(final_df["label"].value_counts())
final_df.head(10)

label
F    16
T    13
U     4
Name: count, dtype: int64


,key,date_sentence,label,_raw,category,evidence_quote,_evi_raw
0,772109,의뢰사유 및 주호소 상기 환자는 교우 문제와 가족 내 갈등으로 우울과 불안...,F,F,,,
1,772109,2018년 2월 27일에 군입대 신체검사를 앞두고 면밀한 평가를 위해서 본원 정신건...,F,F,,,
2,772109,2018년 5월에 병역판정검사를 하여 1급 현역 판정을 받았으며 2020년 8월로 ...,F,F,,,
3,772109,"환자와의 면담에 의하면, 작년 5월에 실시한 병역판정검사에서 1급 현역 판정을 받았...",F,F,,,
4,772109,올해 초부터 교내 기독교 동아리에서 지원해주는 기숙사에서 형들과 공동체 생활을 하고...,U,U,,,
5,772109,어머니도 잔소리가 많은 편으로 올해 들어 학업 문제와 관련해 잔소리를 많이 했고 본...,T,T,잔소리/통제/간섭,잔소리를 많이 했고 본인도 바쁘고 힘든 학기를 보내면서 평상시보다 더 예민하게 받아...,잔소리/통제/간섭 | 잔소리를 많이 했고 본인도 바쁘고 힘든 학기를 보내면서 평상시...
6,6701677,의뢰사유 및 현병력 2019년 12월 22일 전선에 줄을 매고 자살을 시도하여 본원...,T,T,기타,줄을 매고 자살을 시도,기타 | 줄을 매고 자살을 시도
7,6701677,어느 날(1993년) 배우자와 남자가 방에 함께 있는 것을 보고 난 이후부터 외도를...,T,T,의심/질투/감시,외도를 의심함,의심/질투/감시 | 외도를 의심함
8,6701677,2005년까지 아내가 바람을 피웠다고 생각하며 이에 대해 계속 생각나고 화가 나서 ...,T,T,의심/질투/감시,아내가 바람을 피웠다고 생각,의심/질투/감시 | 아내가 바람을 피웠다고 생각
9,6701677,간이정신건강상태검사에서의 2018년~2019년 결과에서 인지기능이 감퇴 되었다는 뚜...,F,F,,,


In [24]:
known = final_df[final_df["label"].isin(["T", "F"])]
t_rate = (known["label"] == "T").mean() if len(known) else float("nan")

print("Total date sentences:", len(final_df))
print("Known(T/F):", len(known))
print("T rate among known:", t_rate)

out_csv = str(Path(OUT_DIR) / "date_sentences_TFU_with_evidence.csv")
final_df.to_csv(out_csv, index=False, encoding="utf-8-sig")
print("saved:", out_csv)

Total date sentences: 33
Known(T/F): 23
T rate among known: 0.4782608695652174
saved: /home/rsm/DATE_OUT/date_sentences_TFU_with_evidence.csv
